In [ ]:
import os
from pyspark import SparkContext, SparkConf

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

conf = (SparkConf().setAppName("MyApp").setMaster("local[*]")
    .set("spark.driver.memory", "4g")
    .set("spark.driver.cores", "2")
    .set("spark.executor.memory", "4g")
    .set("spark.executor.cores", "4")
    .set("spark.executor.instances", "2")
    .set("spark.cores.max", "16")
    .set("spark.dynamicAllocation.enabled", True))

SparkContext.getOrCreate().stop()
sc = SparkContext(conf=conf).getOrCreate()

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
    .appName("MyApp")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

data = [("Abraham", "Apples", 5), 
        ("Jacob", "Bananas", 3),
        ("Jacob", "Oranges", 2),
        ("Benjamin", "Apples", 2),
        ("David", "Bananas", 2),
        ("David", "Oranges", 5),
        ("Esau", "Apples", 3),
        ("Esau", "Bananas", 5),
        ("Ishmael", "Oranges", 4),
        ("Samael", "Apples", 4),
        ("Samael", "Bananas", 1),
        ("Samael", "Oranges", 1)]
columns = ["Customer", "Item", "Quantity"]
df = spark.createDataFrame(data, columns)

In [ ]:
df.show()
print(df.count())
df.printSchema()

In [ ]:
from pyspark.sql.functions import col, sum
# APPROACH 1:
total_apple = df.filter(col("Item") == "Apples").agg(sum("Quantity").alias("Total_Apples")).collect()[0]["Total_Apples"]
print(f"Total Apples sold: {total_apple}")
# APPROACH 2:
total_apple_2 = df.groupBy("Item").agg(sum("Quantity").alias("Total_Quantity")).filter(col("Item") == "Apples").collect()[0]["Total_Quantity"]
print(f"Total Apples sold (approach 2): {total_apple_2}")
# APPROACH 3:
df.createOrReplaceTempView("purchases")
total_apple_3 = spark.sql("SELECT SUM(Quantity) AS Total_Apples FROM purchases WHERE Item = 'Apples'").collect()[0]["Total_Apples"]
print(f"Total Apples sold (approach 3): {total_apple_3}")
# APPROACH 4:
total_apple_4 = spark.sql("SELECT Item, SUM(Quantity) AS Total_Quantity FROM purchases GROUP BY Item HAVING Item = 'Apples'").collect()[0]["Total_Quantity"]
print(f"Total Apples sold (approach 4): {total_apple_4}")


In [ ]:
price_data = [("Apples", 1.0),
              ("Bananas", 0.5),
              ("Oranges", 0.8)]
price_columns = ["Item", "Price"]
price_df = spark.createDataFrame(price_data, price_columns)
price_df.show() # Show table to check content

In [ ]:
from pyspark.sql.functions import expr
# Join purchases with prices on Item column
joined_df = df.join(price_df, on="Item", how="inner")
# Calculate total cost for each purchase
joined_df = joined_df.withColumn("Total_Cost", expr("Quantity * Price"))
# Filter for Jacob's purchases and sum the total cost
jacob_total = joined_df.filter(col("Customer") == "Jacob").agg(sum("Total_Cost").alias("Jacob_Total")).collect()[0]["Jacob_Total"]
print(f"Jacob's total grocery spending: ${jacob_total:.2f}")

In [ ]:
spark.sparkContext._jsc.hadoopConfiguration().set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.4.1") # Library for connecting to AWS S3/MinIO
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://10.1.11.5:9000") # MinIO endpoint (location)
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.access.key", "test") # username
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.secret.key", "hcmuthpcc") # password
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true") # Enable path style access (http://10.1.11.5:9000/big-data /test/... instead of http://big-data.endpoint/test/...)
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # Use basic credentials that we set above
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Use S3A file system
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.committer.name", "magic") # Use magic committer, better than default FileOutputCommitter
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.committer.magic.partitioned.enabled", "true") # Enable partitioned writing
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.fast.upload", "true") # Enable fast upload
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.fast.upload.buffer", "disk") # Use disk buffer for fast upload
# OPTIONAL
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.committer.staging.conflict-mode", "replace") # Overwrite existing files
# OPTIONAL
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.committer.staging.abort.pending.uploads", "true") # Abort pending uploads

In [ ]:
df.write.mode("overwrite").csv("s3a://big-data/test/jacob.csv", header=True)